# SIEWS+ 5.0 — Stage 3: Fire & Smoke Detection
## Transfer Learning with YOLO26n

**Target classes:**
- `0: fire`
- `1: smoke`

**Strategy:** Fine-tune pretrained YOLO26n dari COCO dengan dataset Fire & Smoke khusus.

---
**Cara pakai di Kaggle:**
1. Upload notebook ini ke Kaggle
2. Add dataset: `Fire and Smoke.v14i.yolo26` (upload dari folder `dataset/`)
3. Enable GPU P100/T4
4. Run All → download `best_stage3_new.pt`

## 1. Setup Environment

In [ ]:
import os
import subprocess

# Detect environment
IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB  = 'google.colab' in str(globals().get('__builtins__', ''))
print(f'Kaggle: {IS_KAGGLE} | Colab: {IS_COLAB}')

# Install ultralytics
subprocess.run(['pip', 'install', '-U', 'ultralytics', '-q'], check=True)

import ultralytics
print(f'Ultralytics version: {ultralytics.__version__}')

In [ ]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    DEVICE = '0'
else:
    print('No GPU — using CPU (akan sangat lambat!)')
    DEVICE = 'cpu'

## 2. Konfigurasi Path

In [ ]:
from pathlib import Path

# =====================================================
# SESUAIKAN path dataset di sini
# =====================================================
if IS_KAGGLE:
    # Path dataset Kaggle setelah di-upload
    DATASET_ROOT = Path('/kaggle/input')  # ganti sesuai nama dataset Kaggle Anda
    OUTPUT_DIR   = Path('/kaggle/working')
else:
    # Lokal (Windows)
    DATASET_ROOT = Path('F:/migas-siews/dataset')
    OUTPUT_DIR   = Path('F:/migas-siews/training/runs')

STAGE3_ZIP  = DATASET_ROOT / 'Fire and Smoke.v14i.yolo26.zip'
EXTRACT_DIR = OUTPUT_DIR / 'stage3_dataset'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Dataset zip : {STAGE3_ZIP}')
print(f'Extract dir : {EXTRACT_DIR}')
print(f'ZIP exists  : {STAGE3_ZIP.exists()}')

## 3. Ekstrak & Validasi Dataset

In [ ]:
import zipfile
import shutil

# Ekstrak dataset
if not EXTRACT_DIR.exists():
    print(f'Extracting {STAGE3_ZIP.name} ...')
    with zipfile.ZipFile(STAGE3_ZIP, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print('Extracted!')
else:
    print(f'Already extracted: {EXTRACT_DIR}')

# Tampilkan struktur
for p in sorted(EXTRACT_DIR.rglob('*.yaml')):
    print(f'  YAML: {p}')

In [ ]:
# Cari data.yaml otomatis
yaml_files = list(EXTRACT_DIR.rglob('data.yaml'))
if not yaml_files:
    yaml_files = list(EXTRACT_DIR.rglob('*.yaml'))

print('YAML files found:')
for y in yaml_files:
    print(f'  {y}')

DATA_YAML = yaml_files[0]
print(f'\nUsing: {DATA_YAML}')

# Tampilkan isi YAML
print('\n--- YAML content ---')
print(DATA_YAML.read_text())

In [ ]:
# Hitung jumlah gambar per split
for split in ['train', 'valid', 'val', 'test']:
    img_dir = EXTRACT_DIR / split / 'images'
    if img_dir.exists():
        imgs = list(img_dir.glob('*.*'))
        lbl_dir = EXTRACT_DIR / split / 'labels'
        lbls = list(lbl_dir.glob('*.txt')) if lbl_dir.exists() else []
        print(f'[{split:6s}] images={len(imgs):5d}  labels={len(lbls):5d}')

## 4. Buat YAML Custom (Fix Path)

In [ ]:
import yaml

# Deteksi nama split val (bisa 'valid' atau 'val')
val_split = 'valid' if (EXTRACT_DIR / 'valid' / 'images').exists() else 'val'

# Satu sumber kebenaran untuk class order.
# Roboflow label: 0 = fire, 1 = smoke. Jangan ubah urutan ini kecuali label audit membuktikan sebaliknya.
CANONICAL_CLASS_NAMES = {0: 'fire', 1: 'smoke'}
CANONICAL_CLASS_COLORS = {0: (255, 80, 0), 1: (150, 150, 150)}  # RGB: fire=orange, smoke=gray

# Buat YAML baru dengan path absolut yang benar
custom_yaml = OUTPUT_DIR / 'stage3_fire_smoke.yaml'
yaml_content = {
    'path': str(EXTRACT_DIR),
    'train': 'train/images',
    'val':   f'{val_split}/images',
    'test':  'test/images' if (EXTRACT_DIR / 'test' / 'images').exists() else f'{val_split}/images',
    'nc': 2,
    'names': CANONICAL_CLASS_NAMES,
}

with open(custom_yaml, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print(f'Custom YAML: {custom_yaml}')
print(custom_yaml.read_text())

## 5. Visualisasi Sample Dataset

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random

COLORS = CANONICAL_CLASS_COLORS
NAMES  = CANONICAL_CLASS_NAMES

train_img_dir = EXTRACT_DIR / 'train' / 'images'
train_lbl_dir = EXTRACT_DIR / 'train' / 'labels'
samples = random.sample(list(train_img_dir.glob('*.jpg')) + list(train_img_dir.glob('*.png')), min(6, len(list(train_img_dir.glob('*.*')))))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flat, samples):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl_path = train_lbl_dir / (img_path.stem + '.txt')
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            cls, xc, yc, bw, bh = map(float, line.split())
            cls = int(cls)
            x1 = int((xc - bw/2) * w); y1 = int((yc - bh/2) * h)
            x2 = int((xc + bw/2) * w); y2 = int((yc + bh/2) * h)
            color = COLORS.get(cls, (0,255,0))
            cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
            cv2.putText(img, f'ID:{cls} {NAMES.get(cls, "?")}', (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    ax.imshow(img); ax.axis('off'); ax.set_title(img_path.name[:20])

plt.suptitle('Sample Dataset — Fire & Smoke', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'dataset_samples.png'), dpi=100)
plt.show()
print('Samples visualized!')

## 6. Load Base Model YOLO26n

In [ ]:
from ultralytics import YOLO

# Download pretrained YOLO26n (COCO, 80 classes)
# YOLO26 = end-to-end model tanpa NMS — lebih cepat inferensi
base_model = YOLO('yolo26n.pt')

print(f'Model loaded: {base_model.model_name}')
print(f'Base classes: {len(base_model.names)} (COCO)')
print('\nSiap untuk fine-tuning ke 2 kelas: fire, smoke')

## 7. Training (Transfer Learning)

### Penjelasan Hyperparameter:
| Parameter | Nilai | Alasan |
|---|---|---|
| `epochs` | 80 | Cukup untuk konvergensi fire/smoke |
| `batch` | 16 | Sesuai VRAM GPU Kaggle |
| `imgsz` | 640 | Resolusi standar YOLO |
| `patience` | 15 | Early stopping jika tidak improve |
| `freeze` | 10 | Freeze 10 layer pertama — transfer learning |
| `hsv_h/s/v` | tinggi | Fire/smoke sangat bergantung warna |
| `mosaic` | 1.0 | Gabungkan 4 gambar — perbanyak variasi |
| `flipud` | 0.1 | Api bisa dari segala arah |
| `optimizer` | `AdamW` | Lebih stabil untuk fine-tuning |

In [ ]:
# =====================================================
# KONFIGURASI TRAINING — sesuaikan jika perlu
# =====================================================
EPOCHS   = 80
BATCH    = 16   # kurangi ke 8 jika OOM (out of memory)
IMGSZ    = 640
PATIENCE = 15   # early stopping
FREEZE   = 10   # freeze backbone layers — transfer learning

print(f'Config: epochs={EPOCHS}, batch={BATCH}, imgsz={IMGSZ}, device={DEVICE}')
print(f'        freeze={FREEZE} layers (transfer learning mode)')

In [ ]:
results = base_model.train(
    data    = str(custom_yaml),
    epochs  = EPOCHS,
    batch   = BATCH,
    imgsz   = IMGSZ,
    device  = DEVICE,
    patience= PATIENCE,
    freeze  = FREEZE,          # transfer learning: bekukan backbone
    pretrained = True,

    # Project output
    project  = str(OUTPUT_DIR / 'runs'),
    name     = 'stage3_fire_smoke',
    exist_ok = True,

    # Augmentasi — penting untuk fire/smoke
    augment  = True,
    mosaic   = 1.0,          # 4-image mosaic
    mixup    = 0.1,          # mix 2 gambar
    copy_paste = 0.1,        # copy-paste objek ke gambar lain
    hsv_h    = 0.03,         # variasi hue (warna api berbeda-beda)
    hsv_s    = 0.9,          # variasi saturasi (asap tebal/tipis)
    hsv_v    = 0.5,          # variasi brightness (siang/malam)
    fliplr   = 0.5,          # flip horizontal
    flipud   = 0.1,          # flip vertikal (api dari berbagai sudut)
    degrees  = 5.0,          # rotasi kecil
    scale    = 0.6,          # variasi skala objek
    translate= 0.1,
    erasing  = 0.3,          # random erasing — robustness

    # Optimizer
    optimizer     = 'AdamW', # lebih stabil untuk fine-tuning
    lr0           = 0.001,   # learning rate awal — rendah untuk fine-tuning
    lrf           = 0.01,    # learning rate akhir (cosine decay)
    weight_decay  = 0.0005,
    warmup_epochs = 3,

    # Save & logging
    save_period = 10,
    verbose     = True,
    plots       = True,
)

## 8. Evaluasi Hasil Training

In [ ]:
# Tampilkan metrik training
metrics = results.results_dict

print('=' * 50)
print('HASIL TRAINING — Stage 3 Fire/Smoke')
print('=' * 50)
map50    = metrics.get('metrics/mAP50(B)', 0)
map5095  = metrics.get('metrics/mAP50-95(B)', 0)
prec     = metrics.get('metrics/precision(B)', 0)
rec      = metrics.get('metrics/recall(B)', 0)

print(f'mAP50       : {map50:.4f}  (target: > 0.85)')
print(f'mAP50-95    : {map5095:.4f}')
print(f'Precision   : {prec:.4f}  (target: > 0.80)')
print(f'Recall      : {rec:.4f}  (target: > 0.80)')
print()

if map50 >= 0.85:
    print('✅ Model sudah baik! mAP50 >= 0.85')
elif map50 >= 0.70:
    print('⚠️  Model cukup baik. Coba tambah epochs atau lebih banyak data.')
else:
    print('❌ Model kurang baik. Lihat saran di bawah.')

In [ ]:
# Tampilkan grafik training (loss curves, mAP)
run_dir = OUTPUT_DIR / 'runs' / 'stage3_fire_smoke'

plot_files = [
    run_dir / 'results.png',
    run_dir / 'confusion_matrix.png',
    run_dir / 'PR_curve.png',
]

from IPython.display import Image, display
for pf in plot_files:
    if pf.exists():
        print(f'--- {pf.name} ---')
        display(Image(str(pf), width=800))

## 9. Validasi pada Val Set

In [ ]:
# Load best model dan validasi
best_pt = run_dir / 'weights' / 'best.pt'
print(f'Loading best model: {best_pt}')

best_model = YOLO(str(best_pt))

# Pastikan metadata class pada weight tidak tertukar saat dipakai untuk plot/inference.
loaded_names = {int(k): v for k, v in dict(best_model.names).items()}
if loaded_names != CANONICAL_CLASS_NAMES:
    print(f'WARNING: model.names berbeda: {loaded_names} -> diset ulang ke {CANONICAL_CLASS_NAMES}')
    best_model.names = CANONICAL_CLASS_NAMES
else:
    print(f'Class order OK: {loaded_names}')

val_results = best_model.val(data=str(custom_yaml), device=DEVICE, verbose=True)

print(f'\nVal mAP50   : {val_results.box.map50:.4f}')
print(f'Val mAP50-95: {val_results.box.map:.4f}')

## 10. Inference Test — Uji Langsung pada Gambar

In [ ]:
# Ambil beberapa gambar test dan jalankan inferensi
# Overlay dibuat manual supaya label selalu mengikuti CANONICAL_CLASS_NAMES,
# bukan metadata plot otomatis yang kadang tersisa dari weight/run lama.
test_img_dir = EXTRACT_DIR / 'test' / 'images'
if not test_img_dir.exists():
    test_img_dir = EXTRACT_DIR / val_split / 'images'

test_imgs = random.sample(
    list(test_img_dir.glob('*.jpg')) + list(test_img_dir.glob('*.png')),
    min(6, len(list(test_img_dir.glob('*.*'))))
)

TEST_CONF = 0.15  # smoke sering lebih diffuse; pakai rendah untuk audit visual
PRED_COLORS_BGR = {0: (0, 80, 255), 1: (160, 160, 160)}  # cv2 BGR

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flat, test_imgs):
    pred = best_model.predict(str(img_path), conf=TEST_CONF, verbose=False)[0]
    img_bgr = cv2.imread(str(img_path))
    boxes = pred.boxes
    label_parts = []

    if boxes is not None and len(boxes) > 0:
        for box in boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            name = CANONICAL_CLASS_NAMES.get(cls_id, f'cls_{cls_id}')
            color = PRED_COLORS_BGR.get(cls_id, (0, 255, 0))
            cv2.rectangle(img_bgr, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img_bgr, f'ID:{cls_id} {name} {conf:.0%}', (x1, max(y1 - 6, 18)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            label_parts.append(f'ID:{cls_id} {name}:{conf:.0%}')

    annotated = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated)
    ax.axis('off')
    preds_str = ', '.join(label_parts) if label_parts else 'no detection'
    ax.set_title(preds_str[:35], fontsize=9)

plt.suptitle('Inference Test — Best Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'inference_test.png'), dpi=100)
plt.show()

## 11. Simpan Model ke Backend

In [ ]:
import shutil

best_pt = run_dir / 'weights' / 'best.pt'
last_pt = run_dir / 'weights' / 'last.pt'

# Simpan ulang best model dengan metadata class yang sudah dipaksa benar.
export_model = best_model if 'best_model' in globals() else YOLO(str(best_pt))
export_model.names = CANONICAL_CLASS_NAMES

if IS_KAGGLE:
    # Di Kaggle: salin ke /kaggle/working agar bisa di-download
    out_best = Path('/kaggle/working/best_stage3_new.pt')
    out_last = Path('/kaggle/working/last_stage3_new.pt')
    export_model.save(str(out_best))
    shutil.copy2(last_pt, out_last)
    print(f'✅ Saved (Kaggle output):')
    print(f'   {out_best}  ({out_best.stat().st_size/1e6:.1f} MB)')
    print(f'   {out_last}  ({out_last.stat().st_size/1e6:.1f} MB)')
    print()
    print('Langkah selanjutnya:')
    print('  1. Download best_stage3_new.pt dari Output tab Kaggle')
    print('  2. Salin ke: f:/migas-siews/backend/models/best_stage3_new.pt')
    print('  3. Restart backend')
else:
    # Lokal: salin langsung ke backend/models
    models_dir = Path('F:/migas-siews/backend/models')
    models_dir.mkdir(parents=True, exist_ok=True)
    out_best = models_dir / 'best_stage3_new.pt'
    export_model.save(str(out_best))
    print(f'✅ Saved to backend: {out_best}  ({out_best.stat().st_size/1e6:.1f} MB)')

## 12. Saran jika mAP Rendah

| Masalah | Solusi |
|---|---|
| mAP50 < 0.70 | Tambah data dari Roboflow: cari `fire detection` / `smoke detection` |
| Banyak false positive | Tambah gambar negatif (tanpa api/asap) ke dataset |
| Detection NONE saat test | Turunkan `conf` threshold ke 0.15 |
| Training lambat | Kurangi `batch=8`, atau upgrade ke GPU T4 x2 di Kaggle |
| Loss tidak turun | Kurangi `lr0=0.0005`, naikkan `warmup_epochs=5` |
| Overfitting | Naikkan `weight_decay=0.001`, tambah `dropout=0.1` |

### Dataset tambahan yang recommended (gratis di Roboflow):
- `D-Fire Dataset` — 21k gambar api & asap outdoor
- `Fire Detection v2` — 5k gambar indoor & outdoor  
- `Wildfire Smoke` — khusus asap hutan